In [1]:
# ===================================================================
# 0. Dependency Imports
# ===================================================================
import os
import argparse
import pickle
import glob
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Check Whether mamba-ssm Is Installed
try:
    from mamba_ssm import Mamba
except ImportError:
    print("="*80)
    print("ERROR: mamba-ssm is not installed. Please install it first:")
    print("pip install causal-conv1d>=1.0.0 mamba-ssm")
    print("="*80)
    raise

# ===================================================================
# 1. Model and Dataset Definitions (Updated for the New Model)
# ===================================================================

# --- Encoder Helper Modules ---
class DownsampleBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride):
        super(DownsampleBlock, self).__init__()
        padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size, stride=stride, padding=padding),
            nn.BatchNorm1d(out_channels),
            nn.GELU()
        )
    def forward(self, x):
        return self.block(x)

class DeepFeatureResBlock(nn.Module):
    def __init__(self, channels, kernel_size=7):
        super().__init__()
        padding = (kernel_size - 1) // 2
        self.block = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size, stride=1, padding=padding),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Conv1d(channels, channels, kernel_size, stride=1, padding=padding),
            nn.BatchNorm1d(channels),
        )
        self.activation = nn.GELU()
    def forward(self, x):
        return self.activation(x + self.block(x))

# --- Hybrid Encoder ---
class HybridCNNMambaEncoder(nn.Module):
    def __init__(self, in_channels, d_model=128, num_cnn_blocks=3, num_mamba_layers=3, is_sequence=True):
        super().__init__()
        self.is_sequence = is_sequence
        if self.is_sequence:
            self.downsampler = nn.Sequential(
                DownsampleBlock(in_channels, 48, kernel_size=5, stride=5),
                DownsampleBlock(48, 96, kernel_size=5, stride=5),
                DownsampleBlock(96, d_model, kernel_size=5, stride=5),
                DownsampleBlock(d_model, d_model, kernel_size=5, stride=4),
                DownsampleBlock(d_model, d_model, kernel_size=5, stride=4)
            )
        else:
            self.downsampler = nn.Sequential(
                nn.Conv1d(in_channels, d_model, kernel_size=1),
                nn.GELU()
            )
        self.cnn_feature_extractor = nn.Sequential(
            *[DeepFeatureResBlock(channels=d_model, kernel_size=7) for _ in range(num_cnn_blocks)]
        )
        self.mamba_layers = nn.ModuleList(
            [Mamba(d_model=d_model, d_state=16, d_conv=4, expand=2) for _ in range(num_mamba_layers)]
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.downsampler(x)
        x = self.cnn_feature_extractor(x)
        x = x.transpose(1, 2)
        x = self.norm(x)
        for mamba_layer in self.mamba_layers:
            x_fw = mamba_layer(x)
            x_rev = torch.flip(x, dims=[1])
            x_bw = mamba_layer(x_rev)
            x_bw = torch.flip(x_bw, dims=[1])
            x = x + x_fw + x_bw
        x = x.transpose(1, 2)
        return x

# --- Fusion Module ---
class SimpleFusion(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.fusion_conv = nn.Sequential(
            nn.Conv1d(in_channels * 2, out_channels, kernel_size=1),
            nn.BatchNorm1d(out_channels),
            nn.GELU()
        )
    def forward(self, seq_emb, omics_emb):
        combined = torch.cat([seq_emb, omics_emb], dim=1)
        fused = self.fusion_conv(combined)
        return fused

# --- 2D Residual Block (Used by the New Decoder) ---
class ConvResBlock(nn.Module):
    def __init__(self, channels, kernel_size=3):
        super().__init__()
        padding = (kernel_size - 1) // 2
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size, padding=padding),
            nn.BatchNorm2d(channels),
            nn.GELU(),
            nn.Conv2d(channels, channels, kernel_size, padding=padding),
            nn.BatchNorm2d(channels),
        )
        self.activation = nn.GELU()
    def forward(self, x):
        res_out = self.block(x)
        return self.activation(x + res_out)

# --- New Decoder: PracticalOuterProductDecoder ---
class PracticalOuterProductDecoder(nn.Module):
    def __init__(self, in_channels=128, bottleneck_channels=16, num_blocks=4):
        super().__init__()
        print(f"🚀 Initializing PracticalOuterProductDecoder with bottleneck channels: {bottleneck_channels}")
        self.bottleneck = nn.Conv1d(in_channels, bottleneck_channels, kernel_size=1)
        initial_2d_channels = bottleneck_channels * 2
        self.initial_conv = nn.Conv2d(initial_2d_channels, 64, kernel_size=1)
        self.res_blocks = nn.Sequential(
            *[ConvResBlock(64, kernel_size=3) for _ in range(num_blocks)]
        )
        self.final_conv = nn.Conv2d(64, 1, kernel_size=3, padding=1)

    def forward(self, fused_embedding):
        bottleneck_emb = self.bottleneck(fused_embedding)
        B, C_small, L = bottleneck_emb.shape
        x1 = bottleneck_emb.unsqueeze(3).expand(-1, -1, L, L)
        x2 = bottleneck_emb.unsqueeze(2).expand(-1, -1, L, L)
        outer_product = torch.cat([x1, x2], dim=1)
        x = self.initial_conv(outer_product)
        x = self.res_blocks(x)
        x = self.final_conv(x)
        x = (x + x.transpose(-2, -1)) / 2
        return x

# --- Final Model: CNNMambaHybridModel ---
class CNNMambaHybridModel(nn.Module):
    def __init__(self, seq_in_channels=4, num_omics_features=4, d_model=128, fusion_out_dim=128,
                 num_cnn_blocks=3, num_mamba_layers=3,
                 decoder_bottleneck_channels=16, decoder_num_blocks=4):
        super().__init__()
        self.seq_encoder = HybridCNNMambaEncoder(
            in_channels=seq_in_channels, d_model=d_model, num_cnn_blocks=num_cnn_blocks, num_mamba_layers=num_mamba_layers, is_sequence=True
        )
        self.omics_encoder = HybridCNNMambaEncoder(
            in_channels=num_omics_features, d_model=d_model, num_cnn_blocks=num_cnn_blocks, num_mamba_layers=num_mamba_layers, is_sequence=False
        )
        self.fusion_module = SimpleFusion(
            in_channels=d_model, out_channels=fusion_out_dim
        )
        self.decoder = PracticalOuterProductDecoder(
            in_channels=fusion_out_dim,
            bottleneck_channels=decoder_bottleneck_channels,
            num_blocks=decoder_num_blocks
        )

    def forward(self, seq_one_hot, omics_signals):
        seq_embedding = self.seq_encoder(seq_one_hot)
        omics_embedding = self.omics_encoder(omics_signals)
        fused_embedding = self.fusion_module(seq_embedding, omics_embedding)
        pred_hic = self.decoder(fused_embedding)
        return pred_hic.squeeze(1)

# --- Dataset Class (Consistent with the Existing Version) ---
class HiCDataAndOmicsDataset(Dataset):
    def __init__(self, pkl_files):
        self.file_paths = pkl_files
        print(f"Dataset initialized with {len(self.file_paths)} multi-modal samples.")
    def __len__(self):
        return len(self.file_paths)
    def __getitem__(self, idx):
        with open(self.file_paths[idx], 'rb') as f:
            data = pickle.load(f)
        sequence = torch.from_numpy(data['sequence_one_hot'].astype(np.float32))
        hic_matrix = torch.log2(torch.from_numpy(data['hic_matrix'].astype(np.float32)) + 1)
        omics_signals = torch.from_numpy(data['omics_signals'].astype(np.float32))
        return sequence, omics_signals, hic_matrix

# ===================================================================
# 2. Core Visualization Function (Retained from the Original Plotting Code)
# ===================================================================
def visualize_single_prediction(model, test_dataset, device, output_dir, index_to_plot):
    """
    Select a sample at the specified test-set index, run inference, and visualize the prediction with the ground truth, CTCF, and DNase signals.
    """
    model.eval()
    print(f"--- Starting comparison visualization for sample index {index_to_plot} ---")

    # 1. Select the specified test sample
    if not test_dataset or index_to_plot >= len(test_dataset):
        print(f"Error: index {index_to_plot} is outside the test dataset range (size: {len(test_dataset)}).")
        return

    # Extract data and metadata
    seq, omics, real_hic = test_dataset[index_to_plot]
    file_path = test_dataset.file_paths[index_to_plot]

    # Parse region information from the filename
    filename = os.path.basename(file_path)
    try:
        chrom, start_str, end_str = filename.replace('.pkl', '').split('_')
        start_mb = int(start_str) / 1_000_000
        end_mb = int(end_str) / 1_000_000
        title = f"GM12878\n{chrom}: {start_mb:.2f}MB - {end_mb:.2f}MB"
        safe_name = f"{chrom}_{start_str}_{end_str}.png"
    except ValueError:
        title = f"Sample: {filename}"
        safe_name = filename.replace('.pkl', '.png')

    print(f"Visualization region: {title}")

    # 2. Run model inference
    with torch.no_grad():
        seq_tensor = seq.unsqueeze(0).to(device)
        omics_tensor = omics.unsqueeze(0).to(device)
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            pred_hic_tensor = model(seq_tensor, omics_tensor)

    # 3. Post-process data for plotting
    pred_hic_np = pred_hic_tensor.squeeze().cpu().numpy()
    real_hic_np = real_hic.squeeze().cpu().numpy()
    
    # Extract CTCF and DNase signals from the omics tensor
    # Assumption: CTCF is channel 0 and DNase is channel 1 (verify against the preprocessing script)
    ctcf_signal = omics[0, :].cpu().numpy()
    dnase_signal = omics[1, :].cpu().numpy()

    # 4. Plot
    vmax = np.percentile(real_hic_np[real_hic_np > 0], 99) if (real_hic_np > 0).any() else 1.0

    fig = plt.figure(figsize=(7, 20))
    gs = gridspec.GridSpec(5, 1,
                           height_ratios=[10, 10, 0.5, 2, 2],
                           hspace=0.2)
    fig.suptitle(title, fontsize=16, y=0.92, fontweight='bold')

    # Panel 1: Predicted Hi-C
    ax0 = plt.subplot(gs[0])
    im_pred = ax0.imshow(pred_hic_np, cmap='Reds', vmin=0, vmax=vmax, aspect='auto')
    ax0.set_title("Predicted Hi-C", fontweight='bold')
    ax0.axis('off')

    # Panel 2: Ground-truth Hi-C
    ax1 = plt.subplot(gs[1])
    im_real = ax1.imshow(real_hic_np, cmap='Reds', vmin=0, vmax=vmax, aspect='auto')
    ax1.set_title("Ground Truth Hi-C", fontweight='bold')
    ax1.axis('off')

    # Color bar
    ax_cbar = plt.subplot(gs[2])
    cbar = fig.colorbar(im_real, cax=ax_cbar, orientation='horizontal')
    cbar.ax.tick_params(labelsize=8)

    # Panel 3: CTCF signal
    ax2 = plt.subplot(gs[3])
    ax2.plot(ctcf_signal, color='#3498db', linewidth=1.5)
    ax2.set_title("CTCF Signal", fontsize=10, y=0.9, fontweight='bold')
    ax2.spines[['top', 'right', 'bottom']].set_visible(False)
    ax2.tick_params(axis='y', labelsize=8)
    ax2.set_xticks([])
    ax2.set_xlim(0, len(ctcf_signal) - 1)

    # Panel 4: DNase signal
    ax3 = plt.subplot(gs[4])
    ax3.plot(dnase_signal, color='#2ecc71', linewidth=1.5)
    ax3.set_title("DNase Signal", fontsize=10, y=0.9, fontweight='bold')
    ax3.spines[['top', 'right', 'bottom']].set_visible(False)
    ax3.tick_params(axis='y', labelsize=8)
    ax3.set_xticks([])
    ax3.set_xlim(0, len(dnase_signal) - 1)

    plt.tight_layout(rect=[0, 0, 1, 0.95])

    # Save the figure
    output_path = os.path.join(output_dir, safe_name)
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

    print(f"Visualization saved to: {output_path}")

# ===================================================================
# 3. Main Execution Logic (NEW)
# ===================================================================
def main(args):
    # --- 1. Set up the environment ---
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"🚀 Found GPU: {torch.cuda.get_device_name(0)}. Using CUDA.")
    else:
        device = torch.device("cpu")
        print("⚠️ CUDA not available. Using CPU.")

    os.makedirs(args.output_dir, exist_ok=True)
    print(f"All visualization images will be saved to: {args.output_dir}")

    # --- 2. Load data ---
    all_files = glob.glob(os.path.join(args.data_dir, "*.pkl"))
    if not all_files:
        raise ValueError(f"Error: no .pkl files were found in '{args.data_dir}'.")
    
    # Define test-set files (for example, use chr7 as the test set)
    test_files = [f for f in all_files if 'chr7' in os.path.basename(f)]
    if not test_files:
        print("Warning: no test files containing 'chr7' in their filenames were found. Check the data and file naming.")
        return
        
    print(f"Found {len(test_files)} test samples.")
    test_dataset = HiCDataAndOmicsDataset(test_files)

    # Determine the number of omics features dynamically from the data
    try:
        with open(test_files[0], 'rb') as f:
            sample_data = pickle.load(f)
            num_omics_features = sample_data['omics_signals'].shape[0]
            print(f"Automatically detected {num_omics_features} omics features from the data file.")
    except Exception as e:
        print(f"Could not automatically detect the number of omics features; error: {e}. Using the default value of 4.")
        num_omics_features = 4

    # --- 3. Initialize the model ---
    print("\n" + "="*80)
    print(f"Initializing the new CNN-Mamba hybrid model...")
    print(f"Model hyperparameters: d_model={args.d_model}, cnn_blocks={args.num_cnn_blocks}, mamba_layers={args.num_mamba_layers}")
    print(f"Decoder hyperparameters: bottleneck={args.decoder_bottleneck_channels}, res_blocks={args.decoder_num_blocks}")
    print("="*80 + "\n")

    model = CNNMambaHybridModel(
        seq_in_channels=4,
        num_omics_features=num_omics_features,
        d_model=args.d_model,
        fusion_out_dim=args.fusion_out_dim,
        num_cnn_blocks=args.num_cnn_blocks,
        num_mamba_layers=args.num_mamba_layers,
        decoder_bottleneck_channels=args.decoder_bottleneck_channels,
        decoder_num_blocks=args.decoder_num_blocks
    )

    # --- 4. Load model weights ---
    if not os.path.exists(args.model_path):
        raise FileNotFoundError(f"Error: model weights were not found at '{args.model_path}'")

    print(f"Loading pretrained weights from '{args.model_path}'...")
    state_dict = torch.load(args.model_path, map_location=device)
    
    # Handle a DataParallel-wrapped model if the 'module.' prefix is present
    if all(key.startswith('module.') for key in state_dict.keys()):
        print("Detected a DataParallel model; removing the 'module.' prefix.")
        state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
        
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    print("Model weights loaded successfully!")

    # --- 5. Generate visualization images ---
    num_to_visualize = min(args.num_plots, len(test_dataset))
    print(f"\nRandomly selecting {num_to_visualize} samples for visualization...")
    
    # Randomly select unique indices
    plot_indices = random.sample(range(len(test_dataset)), k=num_to_visualize)
    
    for i in tqdm(plot_indices, desc="Generating plots"):
        visualize_single_prediction(
            model=model,
            test_dataset=test_dataset,
            device=device,
            output_dir=args.output_dir,
            index_to_plot=i
        )
        
    print(f"\n--- All tasks completed! ---")



In [ ]:
# Configuration Parameters (Set Directly in the Notebook)
args = {
    'data_dir': '/home/jyzhu/proj/bulk_hic_prediction/preprocessed_data_omic_512',  # Preprocessed data directory
    'model_path': '/home/jyzhu/proj/bulk_hic_prediction/plots_practical_outer_product/best_model_practical_outer_product.pth',  # Model weights path
    'output_dir': './final_visualizations',  # Output directory
    'num_plots': 10,  # Number of visualizations
    
    # Model architecture parameters (must match training)
    'd_model': 128,
    'fusion_out_dim': 128,
    'num_cnn_blocks': 3,
    'num_mamba_layers': 3,
    'decoder_bottleneck_channels': 48,
    'decoder_num_blocks': 6
}

# Convert the dictionary to a Namespace object for compatibility
class Args:
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)

args = Args(**args)

# Run the main function
main(args)

🚀 Found GPU: NVIDIA GeForce RTX 3090. Using CUDA.
All visualization images will be saved to: ./final_visualizations
Found 2447 test samples.
Dataset initialized with 2447 multi-modal samples.
Automatically detected 4 omics features from the data file.

Initializing the new CNN-Mamba hybrid model...
Model hyperparameters: d_model=128, cnn_blocks=3, mamba_layers=3
Decoder hyperparameters: bottleneck=48, res_blocks=6

🚀 Initializing PracticalOuterProductDecoder with bottleneck channels: 48
Loading pretrained weights from '/home/jyzhu/proj/bulk_hic_prediction/plots_practical_outer_product/best_model_practical_outer_product.pth'...
Model weights loaded successfully!

Randomly selecting 10 samples for visualization...


Generating plots:   0%|          | 0/10 [00:00<?, ?it/s]

--- Starting comparison visualization for sample index 2058 ---
Visualization region: GM12878
chr7: 60.29MB - 61.31MB


/tmp/ipykernel_3311495/279123103.py:236: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
/tmp/ipykernel_3311495/279123103.py:292: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 1, 0.95])
Generating plots:  10%|█         | 1/10 [00:01<00:13,  1.54s/it]

Visualization saved to: ./final_visualizations/chr7_60288000_61312000.png
--- Starting comparison visualization for sample index 2249 ---
Visualization region: GM12878
chr7: 157.76MB - 158.78MB


...

Visualization saved to: ./final_visualizations/chr7_112768000_113792000.png
--- Starting comparison visualization for sample index 2145 ---
Visualization region: GM12878
chr7: 69.38MB - 70.40MB


Generating plots:  90%|█████████ | 9/10 [00:09<00:01,  1.06s/it]

Visualization saved to: ./final_visualizations/chr7_69376000_70400000.png
--- Starting comparison visualization for sample index 1133 ---
Visualization region: GM12878
chr7: 77.44MB - 78.46MB


Generating plots: 100%|██████████| 10/10 [00:10<00:00,  1.09s/it]

Visualization saved to: ./final_visualizations/chr7_77440000_78464000.png

--- All tasks completed! ---
